# Content-Based Movie Recommender — Data Pipeline

**Author:** Mohammad Idrees Khan
**Dataset:** [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) (Kaggle)

## Objective
Build a content-based recommendation engine that suggests movies similar to a
given title, using only metadata (overview, genres, keywords, cast, crew) —
no user ratings or collaborative signal required. This makes it robust to the
cold-start problem (works even for movies with no rating history).

## Approach
1. Merge movie metadata with credits (cast/crew).
2. Engineer a single `tag` field per movie by concatenating overview, genres,
   keywords, top-3 cast members, and director.
3. Normalize text (lowercase, strip spaces inside multi-word names so
   "Sam Worthington" becomes one token, stem words to reduce vocabulary size).
4. Vectorize tags with `CountVectorizer` (bag-of-words, top 5000 features).
5. Compute pairwise cosine similarity between all movie vectors.
6. Recommend the top-5 most similar movies for a given title.

## Output artifacts
- `movie_dict.pkl` — movie metadata (id, title) as a dict, loaded by the Streamlit app.
- `similarity.pkl` — precomputed cosine similarity matrix.


## 1. Load and merge raw data

Both CSVs come from the Kaggle TMDB dataset and share a `title` column, which we use to join movie metadata with cast/crew credits.

In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv("/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv")
credits = pd.read_csv("/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv")

print(movies.shape, credits.shape)
movies.head(2)

In [ ]:
movies = movies.merge(credits, on="title")
print("After merge:", movies.shape)

## 2. Select the columns relevant to content-based similarity

We drop everything that isn't useful for building a 'content tag' (e.g. budget, revenue, release_date) and keep only the fields that describe *what the movie is about*.

In [ ]:
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]

# A handful of rows have a missing overview; content-based similarity needs
# text in every field, so we drop them rather than impute (imputing plot
# summaries isn't meaningful).
print("Missing values before dropna:\n", movies.isna().sum())
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

## 3. Parse stringified JSON columns

`genres`, `keywords`, `cast`, and `crew` are stored as stringified lists of dicts, e.g. `"[{'id': 28, 'name': 'Action'}]"`. We use `ast.literal_eval` to safely parse these back into Python objects (safe unlike `eval`, since it only evaluates literals).

In [ ]:
import ast

def extract_names(json_str):
    """Extract every `name` field from a stringified list of dicts."""
    return [item["name"] for item in ast.literal_eval(json_str)]

movies["genres"] = movies["genres"].apply(extract_names)
movies["keywords"] = movies["keywords"].apply(extract_names)

In [ ]:
def extract_top_cast(json_str, top_n=3):
    """Keep only the top-N billed cast members (billing order in TMDB's
    `cast` field roughly correlates with importance to the plot)."""
    names = []
    for i, item in enumerate(ast.literal_eval(json_str)):
        if i >= top_n:
            break
        names.append(item["name"])
    return names

movies["cast"] = movies["cast"].apply(extract_top_cast)

In [ ]:
def extract_director(json_str):
    """Pull just the director's name from the crew list (many crew roles
    exist per movie; only the director is a strong similarity signal)."""
    for item in ast.literal_eval(json_str):
        if item["job"].lower() == "director":
            return [item["name"]]
    return []

movies["crew"] = movies["crew"].apply(extract_director)

## 4. Normalize text

Two transformations:
1. Split `overview` into a word list so it can later be concatenated with the other list-type fields.
2. Strip internal spaces from multi-word names (e.g. `"Sam Worthington"` -> `"SamWorthington"`) so the vectorizer treats a full name as one token instead of two generic first/last-name tokens that could spuriously match unrelated people.

In [ ]:
movies["overview"] = movies["overview"].apply(lambda x: x.split())

for col in ["genres", "keywords", "cast", "crew"]:
    movies[col] = movies[col].apply(lambda items: [i.replace(" ", "") for i in items])

## 5. Build the combined `tag` field

Concatenate all list-type fields into one bag of words per movie, then collapse to a single lowercase string.

In [ ]:
movies["tags"] = (
    movies["overview"] + movies["genres"] + movies["keywords"]
    + movies["cast"] + movies["crew"]
)

# Work on an explicit copy from here on to avoid pandas' SettingWithCopyWarning,
# since `new_df` is a column-subset (a view) of `movies`.
new_df = movies[["movie_id", "title", "tags"]].copy()
new_df["tags"] = new_df["tags"].apply(lambda tokens: " ".join(tokens).lower())
new_df.head(3)

## 6. Stem words

Stemming reduces words to their root form (e.g. `"loved"`, `"loving"` -> `"love"`) so the vectorizer doesn't treat different tenses of the same word as unrelated features. This shrinks the vocabulary and improves similarity matching. Porter Stemmer needs no external downloads.

In [ ]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text: str) -> str:
    return " ".join(ps.stem(word) for word in text.split())

new_df["tags"] = new_df["tags"].apply(stem)

## 7. Vectorize with CountVectorizer

We use a simple bag-of-words model capped at the 5000 most frequent terms (after removing English stop words). TF-IDF was considered as an alternative -- see the note at the end of this notebook.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(new_df["tags"]).toarray()

print("Vector matrix shape:", vectors.shape)  # (n_movies, 5000)

## 8. Compute cosine similarity

Cosine similarity measures the angle between two movies' vectors, which works well for sparse, high-dimensional bag-of-words data (it's insensitive to raw term-count magnitude, unlike Euclidean distance).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
print("Similarity matrix shape:", similarity.shape)  # (n_movies, n_movies)

## 9. Sanity-check the recommender

Before exporting, verify the logic manually on a few well-known titles so a reviewer can see it's working without having to run the Streamlit app.

In [ ]:
def recommend(title, n=5):
    idx = new_df[new_df["title"] == title].index[0]
    distances = similarity[idx]
    top = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:n + 1]
    return [new_df.iloc[i].title for i, _score in top]

for test_title in ["Avatar", "The Dark Knight", "Iron Man"]:
    print(f"{test_title} ->", recommend(test_title))

## 10. Export artifacts for the Streamlit app

We export the dict (not the DataFrame directly) since DataFrames pickle less portably across pandas versions. `similarity` is cast to `float32` before saving -- it roughly halves file size (important: the full float64 matrix for ~4800 movies is close to GitHub's 100MB single-file limit).

In [ ]:
import pickle

pickle.dump(new_df.to_dict(), open("movie_dict.pkl", "wb"))
pickle.dump(similarity.astype("float32"), open("similarity.pkl", "wb"))

print("Artifacts saved: movie_dict.pkl, similarity.pkl")

## Next steps / possible improvements
- Swap `CountVectorizer` for `TfidfVectorizer` to down-weight common-but-uninformative terms.
- Weight fields differently (e.g. genre and director may be stronger similarity signals than overview text).
- Add a hybrid signal: blend content-based similarity with a popularity or rating prior to avoid recommending obscure/poorly-rated look-alikes.
- Evaluate quality quantitatively (e.g. genre-overlap precision@5 across a sample of titles) rather than relying on spot checks.
